# 01 — Data Collection

Downloads team-level regular-season game logs from the `nba_api` for two five-season blocks — an early era and a modern era — tags each row with its era, and saves the raw logs to `data/raw/`.

## Setup and configuration

Import the libraries, define the two era blocks, and set the output folder. Each season is
mapped to its era so every row can be labelled `early` or `modern` during cleaning.


In [1]:
import time
import random
from pathlib import Path

import pandas as pd
from nba_api.stats.endpoints.leaguegamelog import LeagueGameLog

# Two five-season blocks for the era comparison.
EARLY_SEASONS = ["2007-08", "2008-09", "2009-10", "2010-11", "2011-12"]
MODERN_SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]
ALL_SEASONS = EARLY_SEASONS + MODERN_SEASONS

# Label each season with its era block.
ERA_BY_SEASON = {s: "early" for s in EARLY_SEASONS}
ERA_BY_SEASON.update({s: "modern" for s in MODERN_SEASONS})

# Save raw CSVs under data/raw at the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_TIMEOUT = 60   # seconds allowed for each API call
BASE_SLEEP = 1.5       # base pause between season calls

## Columns to keep

The endpoint returns many columns; only the ones needed for the Four Factors and the join key
are kept. `RENAME` maps the raw names to the final names, and `FINAL_COLS` fixes the column
order in the saved files.


In [2]:
# Raw endpoint column -> final column name. Columns not listed here are dropped.
RENAME = {
    "GAME_ID": "GAME_ID",      # join key for adding opponent stats later
    "GAME_DATE": "Date",
    "TEAM_ABBREVIATION": "Team",
    "WL": "WL",
    "FGM": "FGM",
    "FGA": "FGA",
    "FG3M": "FG3M",
    "FG3A": "FG3A",
    "FTM": "FTM",
    "FTA": "FTA",
    "OREB": "ORB",
    "DREB": "DRB",
    "TOV": "TOV",
    "PTS": "PTS",
}

# Column order for the saved files.
FINAL_COLS = [
    "GAME_ID", "Date", "Season", "Team", "Opp", "Home", "W",
    "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "TOV", "PTS",
    "ORB", "DRB", "era",
]

## Helper functions

Three small functions handle the work for a single season:

- `get_opponent` reads the opponent abbreviation out of the `MATCHUP` text.
- `fetch_season` calls the API and retries with longer waits if a request fails.
- `clean` selects the needed columns and derives `Date`, `Season`, `era`, `W`, `Opp`, and `Home`.


In [3]:
def get_opponent(matchup):
    # MATCHUP reads like "GSW vs. LAL" (home) or "GSW @ LAL" (away).
    separator = " vs. " if "vs." in matchup else " @ "
    return matchup.split(separator)[1].strip()


def fetch_season(season):
    # Try a few times, waiting longer after each failed attempt.
    for wait in [0, 5, 15, 30]:
        if wait:
            time.sleep(wait)
        try:
            log = LeagueGameLog(
                player_or_team_abbreviation="T",
                season_type_all_star="Regular Season",
                season=season,
                timeout=REQUEST_TIMEOUT,
            )
            return log.get_data_frames()[0]
        except Exception:
            continue
    raise RuntimeError(f"Could not download {season} after several attempts")


def clean(raw, season):
    # Keep and rename the needed columns.
    df = raw[list(RENAME)].rename(columns=RENAME)
    df["Date"] = pd.to_datetime(df["Date"])
    df["Season"] = season
    df["era"] = ERA_BY_SEASON[season]
    df["W"] = (df["WL"] == "W").astype(int)

    # Opponent and home flag come from the MATCHUP text.
    df["Opp"] = raw["MATCHUP"].apply(get_opponent)
    df["Home"] = raw["MATCHUP"].apply(lambda m: 1 if "vs." in m else 0)

    return df[FINAL_COLS]

## Download and save each season

Loop over every season, download it, clean it, and save one CSV per season. A season that is
already on disk is skipped, so the notebook can be re-run cheaply. Short pauses between calls
keep within the API rate limit.


In [4]:
for season in ALL_SEASONS:
    out_path = RAW_DIR / f"team_games_{season}.csv"

    # Skip seasons already downloaded so re-runs are cheap.
    if out_path.exists():
        print(f"{season}: already saved")
        continue

    print(f"{season}: downloading...")
    df = clean(fetch_season(season), season)
    df.to_csv(out_path, index=False)
    print(f"{season}: saved {len(df)} rows")

    # Pause between calls; older seasons respond slower, so wait a little longer.
    pause = BASE_SLEEP + random.uniform(0, 1.5)
    if season in EARLY_SEASONS:
        pause += 2
    time.sleep(pause)

2007-08: already saved
2008-09: already saved
2009-10: already saved
2010-11: already saved
2011-12: already saved
2020-21: already saved
2021-22: already saved
2022-23: already saved
2023-24: already saved
2024-25: already saved


## Combine into one file

Read the per-season CSVs back in and concatenate them into a single `team_games_all.csv`, which
the next notebooks use as the starting point.


In [5]:
# Combine every season into one file.
frames = [
    pd.read_csv(RAW_DIR / f"team_games_{s}.csv", dtype={"GAME_ID": str}, parse_dates=["Date"])
    for s in ALL_SEASONS
]
all_games = pd.concat(frames, ignore_index=True)
all_games.to_csv(RAW_DIR / "team_games_all.csv", index=False)
print(f"Combined {len(all_games)} rows into team_games_all.csv")

Combined 23820 rows into team_games_all.csv
